# Pronóstico de oportunidades comerciales

Este notebook desarrolla un sistema de forecasting comercial basado en oportunidades históricas del CRM. El objetivo es estimar la probabilidad de ganar cada oportunidad, el valor que podría generar si se gana y el valor esperado del pipeline.

La evaluación se realizará de forma temporal para simular cómo habría funcionado el sistema sobre oportunidades futuras. Las variables de cierre se conservarán únicamente para construir los objetivos y evaluar los resultados.

In [ ]:
import sys
from pathlib import Path

import matplotlib
#import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn


print("Python version: ", sys.version)
print("Executable: ", sys.executable)
print("Working directory: ", Path.cwd())
print("Numpy: ", np.__version__)
print("Pandas: ", pd.__version__)
print("Scikit-Learn: ", sklearn.__version__)
print("Matplotlib: ", matplotlib.__version__)

Python version:  3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Executable:  c:\Users\hecto\Documents\Proyectos\customer_intelligence\.venv\Scripts\python.exe
Working directory:  c:\Users\hecto\Documents\Proyectos\customer_intelligence\notebooks
Numpy:  2.5.1
Pandas:  3.0.3
Scikit-Learn:  1.9.0
Matplotlib:  3.11.1


## 1. Carga de datos

El dataset histórico contiene oportunidades cerradas y se utilizará para entrenar y evaluar los modelos. El dataset de scoring contiene oportunidades en etapa `Engaging` y se reservará para la predicción operativa final.

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MODELING_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "opportunity_modeling.parquet"

SCORING_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "opportunity_scoring.parquet"

opportunity_modeling = pd.read_parquet(MODELING_DATA_PATH)

opportunity_scoring = pd.read_parquet(SCORING_DATA_PATH)

print("Modeling data path: ", MODELING_DATA_PATH)
print("Modeling shape: ", opportunity_modeling.shape)
print()
print("Scoring data path: ", SCORING_DATA_PATH)
print("Scoring shape: ", opportunity_scoring.shape)
print()
print("Columns: ", opportunity_modeling.columns)

display(opportunity_modeling.head())

Modeling data path:  c:\Users\hecto\Documents\Proyectos\customer_intelligence\data\processed\opportunity_modeling.parquet
Modeling shape:  (6711, 23)

Scoring data path:  c:\Users\hecto\Documents\Proyectos\customer_intelligence\data\processed\opportunity_scoring.parquet
Scoring shape:  (1589, 23)

Columns:  Index(['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage',
       'engage_date', 'close_date', 'close_value', 'is_closed', 'is_won',
       'series', 'sales_price', 'manager', 'regional_office', 'sector',
       'year_established', 'revenue', 'employees', 'office_location',
       'subsidiary_of', 'engage_year', 'engage_month', 'engage_quarter'],
      dtype='str')


,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value,is_closed,is_won,...,regional_office,sector,year_established,revenue,employees,office_location,subsidiary_of,engage_year,engage_month,engage_quarter
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0,True,True,...,Central,retail,2001.0,718.62,2448.0,United States,<NA>,2016,10,4
1,Z063OYW0,Darcel Schlecht,GTX Pro,Isdom,Won,2016-10-25,2017-03-11,4514.0,True,True,...,Central,medical,2002.0,3178.24,4540.0,United States,<NA>,2016,10,4
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0,True,True,...,Central,retail,2001.0,718.62,2448.0,United States,<NA>,2016,10,4
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0,True,True,...,Central,software,1998.0,2714.90,2641.0,United States,Acme Corporation,2016,10,4
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0,True,True,...,West,services,1982.0,792.46,1299.0,United States,<NA>,2016,10,4


## 2. Disponibilidad de datos y variable objetivo

La selección de variables debe respetar el momento de predicción. Solo se podrán utilizar atributos conocidos cuando una oportunidad se encuentra en `Engaging`. Las fechas, valores y estados de cierre se reservarán para construir los objetivos y evaluar los resultados.

In [4]:
TARGET = "is_won"

AUDIT_COLUMNS = [
    "product",
    "series",
    "sales_price",
    "sales_agent",
    "manager",
    "regional_office",
    "account",
    "sector",
    "year_established",
    "revenue",
    "employees",
    "office_location",
    "subsidiary_of"
]

missing_comparison = pd.DataFrame(
    {
        "modeling_missing": opportunity_modeling[AUDIT_COLUMNS].isna().sum(),
        "modeling_missing_pct": (
            opportunity_modeling[AUDIT_COLUMNS].isna().mean() * 100
        ),
        "scoring_missing": opportunity_scoring[AUDIT_COLUMNS].isna().sum(),
        "scoring_missing_pct": (
            opportunity_scoring[AUDIT_COLUMNS].isna().mean() * 100
        )
    }
)

target_summary = (
    opportunity_modeling[TARGET]
    .value_counts()
    .rename(index={True: "Won", False: "Lost"})
    .rename("opportunities")
    .to_frame()
)

target_summary["share_pct"] = (
    target_summary["opportunities"] / len(opportunity_modeling) * 100
)

print("Duplicate modeling IDs: ", opportunity_modeling["opportunity_id"].duplicated().sum())
print("Duplicate scoring IDs: ", opportunity_scoring["opportunity_id"].duplicated().sum())
print("Missing targets: ", opportunity_modeling[TARGET].isna().sum())
print()
print("First engagement: ", opportunity_modeling["engage_date"].min())
print("Last engagement: ", opportunity_modeling["engage_date"].max())
print("First close: ", opportunity_modeling["close_date"].min())
print("Last close: ", opportunity_modeling["close_date"].max())
print()

display(target_summary.round(2))
display(missing_comparison.round(2))

Duplicate modeling IDs:  0
Duplicate scoring IDs:  0
Missing targets:  0

First engagement:  2016-10-20 00:00:00
Last engagement:  2017-12-27 00:00:00
First close:  2017-03-01 00:00:00
Last close:  2017-12-31 00:00:00



,opportunities,share_pct
is_won,,
Won,4238,63.15
Lost,2473,36.85


,modeling_missing,modeling_missing_pct,scoring_missing,scoring_missing_pct
product,0,0.00,0,0.00
series,0,0.00,0,0.00
sales_price,0,0.00,0,0.00
sales_agent,0,0.00,0,0.00
manager,0,0.00,0,0.00
regional_office,0,0.00,0,0.00
account,0,0.00,1088,68.47
sector,0,0.00,1088,68.47
year_established,0,0.00,1088,68.47
revenue,0,0.00,1088,68.47


## 3. Evolución temporal y división de los datos

La evaluación seguirá el orden cronológico de las oportunidades. Los dos primeros trimestres se utilizarán para entrenamiento, el tercer trimestre para selección del modelo y el cuarto trimestre quedará reservado como prueba final.

In [5]:
opportunity_modeling["close_quarter"] = (
    opportunity_modeling["close_date"].dt.to_period("Q")
)

quarter_summary = opportunity_modeling.groupby("close_quarter").agg(
    opportunities=("opportunity_id", "size"),
    won_opportunities=("is_won", "sum"),
    actual_revenue=("close_value", "sum")
)

quarter_summary["lost_opportunities"] = (
    quarter_summary["opportunities"]
    - quarter_summary["won_opportunities"]
)

quarter_summary["win_rate_pct"] = (
    quarter_summary["won_opportunities"]
    / quarter_summary["opportunities"]
    * 100
)

train_data = opportunity_modeling.loc[
    opportunity_modeling["close_quarter"]
    <= pd.Period("2017Q2")
].copy()

validation_data = opportunity_modeling.loc[
    opportunity_modeling["close_quarter"]
    == pd.Period("2017Q3")
].copy()

test_data = opportunity_modeling.loc[
    opportunity_modeling["close_quarter"]
    == pd.Period("2017Q4")
].copy()

split_summary = pd.DataFrame(
    {
        "train": {
            "opportunities": len(train_data),
            "won_opportunities": train_data[TARGET].sum(),
            "win_rate_pct": train_data[TARGET].mean() * 100
        },
        "validation": {
            "opportunities": len(validation_data),
            "won_opportunities": validation_data[TARGET].sum(),
            "win_rate_pct": validation_data[TARGET].mean() * 100
        },
        "test": {
            "opportunities": len(test_data),
            "won_opportunities": test_data[TARGET].sum(),
            "win_rate_pct": test_data[TARGET].mean() * 100
        }
    }
).T

display(quarter_summary.round(2))
display(split_summary.round(2))

,opportunities,won_opportunities,actual_revenue,lost_opportunities,win_rate_pct
close_quarter,,,,,
2017Q1,647,531,1134672.0,116,82.07
2017Q2,2032,1254,3086111.0,778,61.71
2017Q3,2047,1257,2982255.0,790,61.41
2017Q4,1985,1196,2802496.0,789,60.25


,opportunities,won_opportunities,win_rate_pct
train,2679.0,1785.0,66.63
validation,2047.0,1257.0,61.41
test,1985.0,1196.0,60.25


## 4. Selección de variables

El modelo utilizará información disponible durante la etapa `Engaging`: producto, precio de lista, responsable comercial, oficina regional y mes de interacción. Los atributos de cuenta se conservarán para análisis descriptivo, pero no serán necesarios para generar una predicción.

In [6]:
MODEL_CATEGORICAL_FEATURES = [
    "product",
    "sales_agent",
    "regional_office",
    "engage_month"
]

MODEL_NUMERIC_FEATURES = [
    "sales_price"
]

MODEL_FEATURES = (
    MODEL_CATEGORICAL_FEATURES
    + MODEL_NUMERIC_FEATURES
)

PROFILING_ONLY_FEATURES = [
    "series",
    "manager",
    "account",
    "sector",
    "year_established",
    "revenue",
    "employees",
    "office_location",
    "subsidiary_of",
    "engage_year",
    "engage_quarter"
]

LEAKAGE_COLUMNS = {
    "deal_stage",
    "is_closed",
    "is_won",
    "close_date",
    "close_value"
}

if not LEAKAGE_COLUMNS.isdisjoint(MODEL_FEATURES):
    raise ValueError("Model features contain outcome information")

X_train = train_data[MODEL_FEATURES].copy()
y_train = train_data[TARGET].astype("int64")

X_validation = validation_data[MODEL_FEATURES].copy()
y_validation = validation_data[TARGET].astype("int64")

X_test = test_data[MODEL_FEATURES].copy()
y_test = test_data[TARGET].astype("int64")

feature_summary = pd.DataFrame(
    {
        "dtype": opportunity_modeling[MODEL_FEATURES].dtypes,
        "unique_train_values": X_train.nunique(),
        "missing_train": X_train.isna().sum(),
        "missing_validation": X_validation.isna().sum(),
        "missing_test": X_test.isna().sum(),
        "missing_scoring": opportunity_scoring[MODEL_FEATURES].isna().sum()
    }
)

print("Categorical features: ", MODEL_CATEGORICAL_FEATURES)
print("Numeric features: ", MODEL_NUMERIC_FEATURES)
print("Profiling only features: ", PROFILING_ONLY_FEATURES)
print()
print("Train shape: ", X_train.shape)
print("Validation shape: ", X_validation.shape)
print("Test shape: ", X_test.shape)

display(feature_summary)

Categorical features:  ['product', 'sales_agent', 'regional_office', 'engage_month']
Numeric features:  ['sales_price']
Profiling only features:  ['series', 'manager', 'account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of', 'engage_year', 'engage_quarter']

Train shape:  (2679, 5)
Validation shape:  (2047, 5)
Test shape:  (1985, 5)


,dtype,unique_train_values,missing_train,missing_validation,missing_test,missing_scoring
product,string,7,0,0,0,0
sales_agent,string,30,0,0,0,0
regional_office,string,3,0,0,0,0
engage_month,int32,9,0,0,0,0
sales_price,float64,7,0,0,0,0


## 5. Preprocesamiento

Las variables categóricas se convertirán mediante one-hot encoding. El precio de lista se estandarizará para que pueda utilizarse correctamente en modelos lineales. Las categorías desconocidas se ignorarán para que el pipeline pueda puntuar oportunidades futuras.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


categorical_preprocessor = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

numeric_preprocessor = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_preprocessor,
            MODEL_CATEGORICAL_FEATURES
        ),
        (
            "numeric",
            numeric_preprocessor,
            MODEL_NUMERIC_FEATURES
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")

X_train_preprocessed = preprocessor.fit_transform(X_train)

X_validation_preprocessed = preprocessor.transform(X_validation)

X_test_preprocessed = preprocessor.transform(X_test)

X_scoring_preprocessed = preprocessor.transform(
    opportunity_scoring[MODEL_FEATURES]
)

print("Preprocessed train shape: ", X_train_preprocessed.shape)
print("Preprocessed validation shape: ", X_validation_preprocessed.shape)
print("Preprocessed test shape: ", X_test_preprocessed.shape)
print("Preprocessed scoring shape: ", X_scoring_preprocessed.shape)
print()
print(
    "Missing preprocessed values: ",
    int(
        X_train_preprocessed.isna().sum().sum()
        + X_validation_preprocessed.isna().sum().sum()
        + X_test_preprocessed.isna().sum().sum()
        + X_scoring_preprocessed.isna().sum().sum()
    )
)

display(X_train_preprocessed.head())

Preprocessed train shape:  (2679, 50)
Preprocessed validation shape:  (2047, 50)
Preprocessed test shape:  (1985, 50)
Preprocessed scoring shape:  (1589, 50)

Missing preprocessed values:  0


,product_GTK 500,product_GTX Basic,product_GTX Plus Basic,product_GTX Plus Pro,product_GTX Pro,product_MG Advanced,product_MG Special,sales_agent_Anna Snelling,sales_agent_Boris Faz,sales_agent_Cassey Cress,...,engage_month_1,engage_month_2,engage_month_3,engage_month_4,engage_month_5,engage_month_6,engage_month_10,engage_month_11,engage_month_12,sales_price
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.485900
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.936959
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.883537
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.694459
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.694459


## 6. Predicción de la probabilidad de conversión

Se compararán una referencia sin capacidad predictiva, un modelo lineal interpretable y un modelo basado en árboles. La selección considerará discriminación, calidad de clasificación y calidad probabilística.

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import brier_score_loss
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score


candidate_classifiers = {
    "Dummy baseline": DummyClassifier(
        strategy="prior"
    ),
    "Logistic regression": LogisticRegression(
        max_iter=2000
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
}

classifier_evaluation_rows = []
validation_probabilities = {}

for model_name, model in candidate_classifiers.items():
    model.fit(X_train_preprocessed, y_train)

    probabilities = model.predict_proba(
        X_validation_preprocessed
    )[:, 1]

    predictions = (
        probabilities >= 0.50
    ).astype("int64")

    classifier_evaluation_rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(
                y_validation,
                predictions
            ),
            "precision": precision_score(
                y_validation,
                predictions
            ),
            "recall": recall_score(
                y_validation,
                predictions
            ),
            "f1_score": f1_score(
                y_validation,
                predictions
            ),
            "roc_auc": roc_auc_score(
                y_validation,
                probabilities
            ),
            "average_precision": average_precision_score(
                y_validation,
                probabilities
            ),
            "brier_score": brier_score_loss(
                y_validation,
                probabilities
            )
        }
    )

    validation_probabilities[model_name] = probabilities

classifier_evaluation = (
    pd.DataFrame(classifier_evaluation_rows)
    .set_index("model")
    .sort_values("brier_score")
)

classifier_evaluation.round(3)

,accuracy,precision,recall,f1_score,roc_auc,average_precision,brier_score
model,,,,,,,
Dummy baseline,0.614,0.614,1.000,0.761,0.500,0.614,0.240
Random forest,0.610,0.614,0.982,0.756,0.513,0.621,0.250
Logistic regression,0.612,0.613,0.995,0.759,0.518,0.633,0.255


### Estabilidad de los patrones comerciales

Antes de ampliar el modelo se comprobará si las tasas históricas de conversión por producto, agente, región y cuenta se mantienen durante el periodo de validación.

In [9]:
STABILITY_FEATURES = [
    "product",
    "sales_agent",
    "regional_office",
    "manager",
    "account",
    "sector"
]

stability_rows = []

for feature in STABILITY_FEATURES:
    train_rates = train_data.groupby(feature)[TARGET].mean()

    validation_rates = validation_data.groupby(feature)[TARGET].mean()

    common_values = train_rates.index.intersection(
        validation_rates.index
    )

    rate_correlation = train_rates.loc[common_values].corr(
        validation_rates.loc[common_values]
    )

    stability_rows.append(
        {
            "feature": feature,
            "shared_values": len(common_values),
            "train_rate_range_pct": (
                train_rates.max() - train_rates.min()
            ) * 100,
            "validation_rate_range_pct": (
                validation_rates.max() - validation_rates.min()
            ) * 100,
            "rate_correlation": rate_correlation
        }
    )

stability_summary = (
    pd.DataFrame(stability_rows)
    .set_index("feature")
)

product_rate_comparison = pd.concat(
    {
        "train": train_data.groupby("product")[TARGET].mean() * 100,
        "validation": (
            validation_data.groupby("product")[TARGET].mean() * 100
        )
    },
    axis=1
)

display(stability_summary.round(3))
display(product_rate_comparison.round(2))

,shared_values,train_rate_range_pct,validation_rate_range_pct,rate_correlation
feature,,,,
product,7,2.889,10.303,0.562
sales_agent,30,18.156,24.878,0.199
regional_office,3,0.714,2.461,-0.209
manager,6,1.377,8.651,0.206
account,85,39.500,52.632,-0.150
sector,10,4.329,14.161,0.410


,train,validation
product,,
GTK 500,66.67,66.67
GTX Basic,66.34,61.1
GTX Plus Basic,66.92,64.24
GTX Plus Pro,67.96,61.11
GTX Pro,66.74,61.41
MG Advanced,65.07,56.36
MG Special,67.28,63.89


### Control del sobreajuste

La baja estabilidad de las tasas históricas sugiere que el modelo puede estar aprendiendo diferencias accidentales entre agentes y periodos. Se evaluarán distintos niveles de regularización y se seleccionará el que produzca las probabilidades más confiables en validación.

In [10]:
REGULARIZATION_VALUES = [
    0.0001,
    0.001,
    0.01,
    0.1,
    1.0,
    10.0
]

regularization_rows = []
regularized_models = {}

for regularization_value in REGULARIZATION_VALUES:
    model = LogisticRegression(
        C=regularization_value,
        max_iter=2000
    )

    model.fit(
        X_train_preprocessed,
        y_train
    )

    probabilities = model.predict_proba(
        X_validation_preprocessed
    )[:, 1]

    predictions = (
        probabilities >= 0.50
    ).astype("int64")

    regularization_rows.append(
        {
            "C": regularization_value,
            "accuracy": accuracy_score(
                y_validation,
                predictions
            ),
            "precision": precision_score(
                y_validation,
                predictions
            ),
            "recall": recall_score(
                y_validation,
                predictions
            ),
            "f1_score": f1_score(
                y_validation,
                predictions
            ),
            "roc_auc": roc_auc_score(
                y_validation,
                probabilities
            ),
            "average_precision": average_precision_score(
                y_validation,
                probabilities
            ),
            "brier_score": brier_score_loss(
                y_validation,
                probabilities
            )
        }
    )

    regularized_models[regularization_value] = model

regularization_evaluation = (
    pd.DataFrame(regularization_rows)
    .set_index("C")
    .sort_values("brier_score")
)

regularization_evaluation.round(3)

,accuracy,precision,recall,f1_score,roc_auc,average_precision,brier_score
C,,,,,,,
0.0001,0.614,0.614,1.000,0.761,0.517,0.638,0.240
0.0010,0.614,0.614,1.000,0.761,0.518,0.639,0.240
0.0100,0.614,0.614,1.000,0.761,0.519,0.638,0.242
0.1000,0.614,0.614,1.000,0.761,0.518,0.633,0.250
1.0000,0.612,0.613,0.995,0.759,0.518,0.633,0.255
10.0000,0.612,0.613,0.995,0.759,0.518,0.633,0.259


## 7. Estimación del valor condicional

El modelo de valor se entrenará únicamente con oportunidades ganadas. Su objetivo será estimar cuánto podría generar una oportunidad en caso de conversión. Esta predicción se combinará posteriormente con la probabilidad de ganar para calcular el valor esperado.

In [12]:
from sklearn.base import clone


VALUE_TARGET = "close_value"

train_won = train_data.loc[
    train_data[TARGET]
].copy()

validation_won = validation_data.loc[
    validation_data[TARGET]
].copy()

test_won = test_data.loc[
    test_data[TARGET]
].copy()

X_value_train = train_won[MODEL_FEATURES].copy()
y_value_train = train_won[VALUE_TARGET].copy()

X_value_validation = validation_won[MODEL_FEATURES].copy()
y_value_validation = validation_won[VALUE_TARGET].copy()

X_value_test = test_won[MODEL_FEATURES].copy()
y_value_test = test_won[VALUE_TARGET].copy()

value_preprocessor = clone(preprocessor)

value_preprocessor.set_output(transform="pandas")

X_value_train_preprocessed = value_preprocessor.fit_transform(
    X_value_train
)

X_value_validation_preprocessed = value_preprocessor.transform(
    X_value_validation
)

X_value_test_preprocessed = value_preprocessor.transform(
    X_value_test
)

print("Won train opportunities: ", len(train_won))
print("Won validation opportunities: ", len(validation_won))
print("Won test opportunities: ", len(test_won))
print()
print(
    "Value train shape: ",
    X_value_train_preprocessed.shape
)
print(
    "Value validation shape: ",
    X_value_validation_preprocessed.shape
)
print(
    "Value test shape: ",
    X_value_test_preprocessed.shape
)

display(
    y_value_train.describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    ).round(2)
)

Won train opportunities:  1785
Won validation opportunities:  1257
Won test opportunities:  1196

Value train shape:  (1785, 50)
Value validation shape:  (1257, 50)
Value test shape:  (1196, 50)


count     1785.00
mean      2364.58
std       2637.94
min         41.00
25%        522.00
50%       1105.00
75%       4374.00
95%       5682.80
99%       6368.04
max      30288.00
Name: close_value, dtype: float64

### Comparación de modelos de valor

Se comparará una referencia basada en la mediana, un modelo lineal regularizado y Random Forest. Además de las métricas individuales, se medirá la diferencia entre el valor total pronosticado y el valor realmente cerrado.

In [13]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error


candidate_value_models = {
    "Median baseline": DummyRegressor(
        strategy="median"
    ),
    "Ridge regression": Ridge(
        alpha=1.0
    ),
    "Random forest": RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
}

value_evaluation_rows = []
validation_value_predictions = {}

actual_validation_value = y_value_validation.sum()

for model_name, model in candidate_value_models.items():
    model.fit(
        X_value_train_preprocessed,
        y_value_train
    )

    predictions = model.predict(
        X_value_validation_preprocessed
    )

    predictions = np.maximum(
        predictions,
        0
    )

    predicted_validation_value = predictions.sum()

    value_evaluation_rows.append(
        {
            "model": model_name,
            "mae": mean_absolute_error(
                y_value_validation,
                predictions
            ),
            "rmse": root_mean_squared_error(
                y_value_validation,
                predictions
            ),
            "mape": mean_absolute_percentage_error(
                y_value_validation,
                predictions
            ),
            "r2_score": r2_score(
                y_value_validation,
                predictions
            ),
            "predicted_total_value": predicted_validation_value,
            "actual_total_value": actual_validation_value,
            "aggregate_error_pct": (
                predicted_validation_value
                / actual_validation_value
                - 1
            ) * 100
        }
    )

    validation_value_predictions[model_name] = predictions

value_model_evaluation = (
    pd.DataFrame(value_evaluation_rows)
    .set_index("model")
    .sort_values("mae")
)

value_model_evaluation.round(3)

,mae,rmse,mape,r2_score,predicted_total_value,actual_total_value,aggregate_error_pct
model,,,,,,,
Ridge regression,199.308,326.054,0.208,0.983,2995084.068,2982255.0,0.430
Random forest,200.143,341.661,0.086,0.981,2991309.632,2982255.0,0.304
Median baseline,1900.132,2785.979,4.061,-0.261,1388985.000,2982255.0,-53.425


### Evaluación temporal del modelo de valor

Random Forest fue seleccionado por su precisión relativa y por la cercanía entre el valor agregado pronosticado y el valor real. El modelo se reentrenará con los datos de desarrollo y se evaluará sobre el último trimestre.

In [14]:
development_won = pd.concat(
    [
        train_won,
        validation_won
    ],
    ignore_index=True
)

X_value_development = development_won[MODEL_FEATURES].copy()

y_value_development = development_won[VALUE_TARGET].copy()

selected_value_preprocessor = clone(preprocessor)

selected_value_preprocessor.set_output(transform="pandas")

X_value_development_preprocessed = (
    selected_value_preprocessor.fit_transform(
        X_value_development
    )
)

X_value_test_preprocessed = (
    selected_value_preprocessor.transform(
        X_value_test
    )
)

selected_value_model = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

selected_value_model.fit(
    X_value_development_preprocessed,
    y_value_development
)

test_value_predictions = selected_value_model.predict(
    X_value_test_preprocessed
)

test_value_predictions = np.maximum(
    test_value_predictions,
    0
)

actual_test_value = y_value_test.sum()

predicted_test_value = test_value_predictions.sum()

test_value_evaluation = pd.DataFrame(
    {
        "mae": [
            mean_absolute_error(
                y_value_test,
                test_value_predictions
            )
        ],
        "rmse": [
            root_mean_squared_error(
                y_value_test,
                test_value_predictions
            )
        ],
        "mape": [
            mean_absolute_percentage_error(
                y_value_test,
                test_value_predictions
            )
        ],
        "r2_score": [
            r2_score(
                y_value_test,
                test_value_predictions
            )
        ],
        "predicted_total_value": [
            predicted_test_value
        ],
        "actual_total_value": [
            actual_test_value
        ],
        "aggregate_error_pct": [
            (
                predicted_test_value
                / actual_test_value
                - 1
            ) * 100
        ]
    },
    index=["Test"]
)

test_value_evaluation.round(3)

,mae,rmse,mape,r2_score,predicted_total_value,actual_total_value,aggregate_error_pct
Test,198.424,348.307,0.085,0.98,2794575.009,2802496.0,-0.283


## 8. Calibración temporal de la conversión

La regresión logística regularizada se calibrará con el trimestre de validación. Esto permite ajustar las probabilidades al cambio observado en la tasa general de conversión sin utilizar información del periodo de prueba. También se comparará contra una referencia basada en la tasa de éxito más reciente.

In [15]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator


SELECTED_REGULARIZATION = 0.001

selected_conversion_model = regularized_models[
    SELECTED_REGULARIZATION
]

calibrated_conversion_model = CalibratedClassifierCV(
    FrozenEstimator(selected_conversion_model),
    method="sigmoid"
)

calibrated_conversion_model.fit(
    X_validation_preprocessed,
    y_validation
)

uncalibrated_test_probabilities = (
    selected_conversion_model.predict_proba(
        X_test_preprocessed
    )[:, 1]
)

calibrated_test_probabilities = (
    calibrated_conversion_model.predict_proba(
        X_test_preprocessed
    )[:, 1]
)

recent_rate_test_probabilities = np.full(
    len(y_test),
    y_validation.mean()
)

conversion_test_probabilities = {
    "Recent-quarter baseline": recent_rate_test_probabilities,
    "Uncalibrated logistic regression": (
        uncalibrated_test_probabilities
    ),
    "Calibrated logistic regression": (
        calibrated_test_probabilities
    )
}

conversion_test_rows = []

actual_test_wins = y_test.sum()
actual_test_win_rate = y_test.mean()

for model_name, probabilities in conversion_test_probabilities.items():
    predicted_wins = probabilities.sum()

    conversion_test_rows.append(
        {
            "model": model_name,
            "roc_auc": roc_auc_score(
                y_test,
                probabilities
            ),
            "average_precision": average_precision_score(
                y_test,
                probabilities
            ),
            "brier_score": brier_score_loss(
                y_test,
                probabilities
            ),
            "predicted_win_rate_pct": (
                probabilities.mean() * 100
            ),
            "actual_win_rate_pct": (
                actual_test_win_rate * 100
            ),
            "predicted_wins": predicted_wins,
            "actual_wins": actual_test_wins,
            "win_count_error_pct": (
                predicted_wins
                / actual_test_wins
                - 1
            ) * 100
        }
    )

conversion_test_evaluation = (
    pd.DataFrame(conversion_test_rows)
    .set_index("model")
    .sort_values("brier_score")
)

conversion_test_evaluation.round(3)

,roc_auc,average_precision,brier_score,predicted_win_rate_pct,actual_win_rate_pct,predicted_wins,actual_wins,win_count_error_pct
model,,,,,,,,
Calibrated logistic regression,0.544,0.655,0.239,61.673,60.252,1224.204,1196,2.358
Recent-quarter baseline,0.500,0.603,0.240,61.407,60.252,1218.928,1196,1.917
Uncalibrated logistic regression,0.544,0.655,0.244,66.998,60.252,1329.909,1196,11.196


## 9. Forecast de valor esperado

El forecast final combinará la probabilidad calibrada de conversión con el valor condicional estimado por Random Forest. La evaluación principal comparará el valor agregado esperado con el ingreso realmente cerrado durante el trimestre de prueba.

In [16]:
X_test_pipeline_value = test_data[MODEL_FEATURES].copy()

X_test_pipeline_value_preprocessed = (
    selected_value_preprocessor.transform(
        X_test_pipeline_value
    )
)

conditional_test_value_predictions = (
    selected_value_model.predict(
        X_test_pipeline_value_preprocessed
    )
)

conditional_test_value_predictions = np.maximum(
    conditional_test_value_predictions,
    0
)

test_forecast = test_data[
    [
        "opportunity_id",
        "product",
        "sales_agent",
        "regional_office",
        "engage_date",
        "close_date",
        "deal_stage",
        "close_value",
        "sales_price"
    ]
].copy()

test_forecast["win_probability"] = (
    calibrated_test_probabilities
)

test_forecast["predicted_won_value"] = (
    conditional_test_value_predictions
)

test_forecast["expected_value"] = (
    test_forecast["win_probability"]
    * test_forecast["predicted_won_value"]
)

recent_rate_expected_value = (
    recent_rate_test_probabilities
    * conditional_test_value_predictions
)

price_baseline_expected_value = (
    recent_rate_test_probabilities
    * test_data["sales_price"].to_numpy()
)

forecast_methods = {
    "Price and recent-rate baseline": (
        price_baseline_expected_value
    ),
    "ML value and recent-rate baseline": (
        recent_rate_expected_value
    ),
    "Calibrated probability and ML value": (
        test_forecast["expected_value"].to_numpy()
    )
}

actual_test_pipeline_value = test_data["close_value"].sum()

forecast_evaluation_rows = []

for method_name, expected_values in forecast_methods.items():
    predicted_pipeline_value = expected_values.sum()

    forecast_evaluation_rows.append(
        {
            "method": method_name,
            "predicted_pipeline_value": predicted_pipeline_value,
            "actual_pipeline_value": actual_test_pipeline_value,
            "aggregate_error": (
                predicted_pipeline_value
                - actual_test_pipeline_value
            ),
            "aggregate_error_pct": (
                predicted_pipeline_value
                / actual_test_pipeline_value
                - 1
            ) * 100,
            "opportunity_mae": mean_absolute_error(
                test_data["close_value"],
                expected_values
            ),
            "opportunity_rmse": root_mean_squared_error(
                test_data["close_value"],
                expected_values
            )
        }
    )

forecast_evaluation = (
    pd.DataFrame(forecast_evaluation_rows)
    .set_index("method")
)

print(
    "Invalid probabilities: ",
    (
        ~test_forecast["win_probability"].between(0, 1)
    ).sum()
)

print(
    "Negative expected values: ",
    test_forecast["expected_value"].lt(0).sum()
)

display(forecast_evaluation.round(2))

Invalid probabilities:  0
Negative expected values:  0


,predicted_pipeline_value,actual_pipeline_value,aggregate_error,aggregate_error_pct,opportunity_mae,opportunity_rmse
method,,,,,,
Price and recent-rate baseline,2871846.47,2802496.0,69350.47,2.47,1127.12,1741.33
ML value and recent-rate baseline,2868948.96,2802496.0,66452.96,2.37,1128.12,1745.57
Calibrated probability and ML value,2881915.40,2802496.0,79419.40,2.83,1125.40,1745.02


In [17]:
test_forecast["calibrated_price_expected_value"] = (
    test_forecast["win_probability"]
    * test_forecast["sales_price"]
)

calibrated_price_predictions = test_forecast[
    "calibrated_price_expected_value"
]

actual_opportunity_values = test_forecast["close_value"]

calibrated_price_total = calibrated_price_predictions.sum()

calibrated_price_evaluation = pd.DataFrame(
    {
        "predicted_pipeline_value": [
            calibrated_price_total
        ],
        "actual_pipeline_value": [
            actual_test_pipeline_value
        ],
        "aggregate_error": [
            calibrated_price_total
            - actual_test_pipeline_value
        ],
        "aggregate_error_pct": [
            (
                calibrated_price_total
                / actual_test_pipeline_value
                - 1
            ) * 100
        ],
        "opportunity_mae": [
            mean_absolute_error(
                actual_opportunity_values,
                calibrated_price_predictions
            )
        ],
        "opportunity_rmse": [
            root_mean_squared_error(
                actual_opportunity_values,
                calibrated_price_predictions
            )
        ]
    },
    index=["Calibrated probability and price"]
)

final_forecast_comparison = pd.concat(
    [
        forecast_evaluation,
        calibrated_price_evaluation
    ]
)

final_forecast_comparison.round(2)

,predicted_pipeline_value,actual_pipeline_value,aggregate_error,aggregate_error_pct,opportunity_mae,opportunity_rmse
Price and recent-rate baseline,2871846.47,2802496.0,69350.47,2.47,1127.12,1741.33
ML value and recent-rate baseline,2868948.96,2802496.0,66452.96,2.37,1128.12,1745.57
Calibrated probability and ML value,2881915.40,2802496.0,79419.40,2.83,1125.40,1745.02
Calibrated probability and price,2884819.56,2802496.0,82323.56,2.94,1124.40,1740.76


In [18]:
ranked_test_forecast = (
    test_forecast
    .sort_values(
        "win_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

actual_test_wins = ranked_test_forecast[
    "deal_stage"
].eq("Won").sum()

overall_test_win_rate = (
    actual_test_wins
    / len(ranked_test_forecast)
)

ranking_rows = []

for top_opportunity_pct in [10, 20, 30]:
    opportunities_reviewed = int(
        np.ceil(
            len(ranked_test_forecast)
            * top_opportunity_pct
            / 100
        )
    )

    selected_opportunities = ranked_test_forecast.head(
        opportunities_reviewed
    )

    wins_found = selected_opportunities[
        "deal_stage"
    ].eq("Won").sum()

    precision = (
        wins_found
        / opportunities_reviewed
    )

    win_capture = (
        wins_found
        / actual_test_wins
    )

    ranking_rows.append(
        {
            "top_opportunity_pct": top_opportunity_pct,
            "opportunities_reviewed": opportunities_reviewed,
            "wins_found": wins_found,
            "precision_pct": precision * 100,
            "win_capture_pct": win_capture * 100,
            "lift": precision / overall_test_win_rate
        }
    )

ranking_evaluation = (
    pd.DataFrame(ranking_rows)
    .set_index("top_opportunity_pct")
)

ranking_evaluation.round(2)

,opportunities_reviewed,wins_found,precision_pct,win_capture_pct,lift
top_opportunity_pct,,,,,
10,199,143,71.86,11.96,1.19
20,397,268,67.51,22.41,1.12
30,596,391,65.60,32.69,1.09


In [19]:
SELECTED_FORECAST_METHOD = "Calibrated probability and price"

selected_test_forecast = test_forecast.copy()

selected_test_forecast["expected_value"] = (
    selected_test_forecast["win_probability"]
    * selected_test_forecast["sales_price"]
)

selected_test_forecast["forecast_rank"] = (
    selected_test_forecast["expected_value"]
    .rank(
        method="first",
        ascending=False
    )
    .astype("int64")
)

selected_method_evaluation = final_forecast_comparison.loc[
    [SELECTED_FORECAST_METHOD]
].copy()

if selected_test_forecast["win_probability"].between(0, 1).all() is False:
    raise ValueError("Win probabilities must be between 0 and 1")

if selected_test_forecast["expected_value"].lt(0).any():
    raise ValueError("Expected values cannot be negative")

if selected_test_forecast["forecast_rank"].duplicated().any():
    raise ValueError("Forecast ranks must be unique")

print("Selected method: ", SELECTED_FORECAST_METHOD)
print("Opportunities evaluated: ", f"{len(selected_test_forecast):,}")
print(
    "Predicted pipeline value: ",
    f"{selected_test_forecast['expected_value'].sum():,.2f}"
)
print(
    "Actual pipeline value: ",
    f"{selected_test_forecast['close_value'].sum():,.2f}"
)

display(selected_method_evaluation.round(2))

Selected method:  Calibrated probability and price
Opportunities evaluated:  1,985
Predicted pipeline value:  2,884,819.56
Actual pipeline value:  2,802,496.00


,predicted_pipeline_value,actual_pipeline_value,aggregate_error,aggregate_error_pct,opportunity_mae,opportunity_rmse
Calibrated probability and price,2884819.56,2802496.0,82323.56,2.94,1124.4,1740.76
